# FACTS — Phase 2: Semantic Consistency Signals
**Behafarin Emam | be379@drexel.edu**  
Capstone 2 — Embedding-based Hallucination Detection

---

## What this notebook does

We compute two consistency signals across the 5 temperature-varied responses per question:

- **Signal 1 (TF-IDF baseline):** word-overlap similarity — already computed in the preprocessing notebook
- **Signal 2 (LSA semantic embeddings):** meaning-level similarity using Latent Semantic Analysis

We then validate both signals by checking whether hallucination-prone categories (Conspiracies, Paranormal, Fiction) score higher inconsistency than factually stable categories (Science, Health).

**Why LSA instead of a neural sentence encoder?**  
LSA (TF-IDF + SVD) runs entirely on sklearn with no GPU or heavy dependencies. It captures topic-level semantic similarity — not as powerful as a transformer encoder (like `all-MiniLM-L6-v2`), but good enough for a meaningful first signal. In the final report, this will be replaced with a proper sentence encoder run locally.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('Ready.')

## 2. Load Data

We need two files:
- `truthfulqa_results.csv` — raw responses (from Phase 1 data generation)
- `truthfulqa_processed.csv` — categories and TF-IDF baseline scores (from preprocessing notebook)

In [ ]:
# Raw responses
raw = pd.read_csv('truthfulqa_results.csv')
raw['Answer'] = raw['Answer'].astype(str).str.strip()
raw['Answer'] = raw['Answer'].str.replace(r'\s+', ' ', regex=True)

# Processed data with categories and TF-IDF signal
processed = pd.read_csv('truthfulqa_processed.csv')

print(f'Raw responses: {raw.shape}')
print(f'Processed questions: {processed.shape}')
print(f'\nProcessed columns: {processed.columns.tolist()}')
processed.head(3)

## 3. What is an Embedding?

An **embedding** converts text into a vector of numbers — a point in high-dimensional space. 
Texts with similar *meaning* end up close together in that space, even if they use different words.

**Example:**
- "Water boils at 100 degrees" 
- "The boiling point of water is 100°C"

These use almost no overlapping words, but a good embedding puts them very close together.

**Why this matters for our project:**  
TF-IDF similarity only sees word overlap. Embeddings see meaning. 
If the LLM gives 5 responses that all *mean the same thing* but use different words, TF-IDF says "inconsistent" — embeddings correctly say "consistent".

### Our method: LSA (Latent Semantic Analysis)

We use a 2-step process:
1. **TF-IDF:** convert each response to a sparse word-frequency vector (8000 dimensions)
2. **SVD (Truncated):** compress those 8000 dimensions down to 100 semantic dimensions

The result is a 100-dimensional embedding for each response that captures topic and meaning.

In [ ]:
# Step 1: TF-IDF on all 4085 responses
# max_features=8000 keeps the 8000 most informative words
# min_df=1 includes a word even if it appears in just one response
# stop_words removes common words like 'the', 'is', 'and'

print("Step 1: Building TF-IDF matrix...")
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=8000,
    min_df=1
)
tfidf_matrix = vectorizer.fit_transform(raw['Answer'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} responses × {tfidf_matrix.shape[1]} word features")

In [ ]:
# Step 2: SVD compression to 100 semantic dimensions
# n_components=100 means we keep 100 'topics'
# random_state=42 makes results reproducible

print("Step 2: Compressing to semantic embeddings (LSA)...")
svd = TruncatedSVD(n_components=100, random_state=42)
embeddings_raw = svd.fit_transform(tfidf_matrix)

# L2 normalize: scale each vector to length 1
# This makes cosine similarity equivalent to dot product — easier math
embeddings = normalize(embeddings_raw)

print(f"Embeddings shape: {embeddings.shape}")
print(f"  → {embeddings.shape[0]} responses × {embeddings.shape[1]} semantic dimensions")
print(f"Variance explained by 100 dimensions: {svd.explained_variance_ratio_.sum():.1%}")
print("\n(A transformer model like all-MiniLM would explain ~70-80% variance with 384 dims)")

## 4. Compute Per-Question Inconsistency

For each question we have 5 response embeddings. We measure how spread out they are by computing **mean pairwise cosine similarity** across all 10 pairs (C(5,2) = 10 pairs).

- **Cosine similarity = 1.0** → vectors point in the same direction → responses mean the same thing
- **Cosine similarity = 0.0** → vectors are perpendicular → responses have nothing in common

**Inconsistency score = 1 − mean similarity**  
Higher = more inconsistent = higher hallucination risk.

In [ ]:
def compute_inconsistency(group_indices, embeddings):
    """
    Given a list of row indices (one per temperature response),
    compute mean pairwise cosine similarity and return inconsistency = 1 - similarity.
    """
    embs = embeddings[group_indices]          # shape: (5, 100)
    sim_matrix = cosine_similarity(embs)      # shape: (5, 5)
    
    # Extract upper triangle only (avoid self-similarity and duplicates)
    n = len(group_indices)
    pair_sims = [sim_matrix[i][j] for i, j in combinations(range(n), 2)]
    
    mean_sim = np.mean(pair_sims)
    std_sim  = np.std(pair_sims)
    return mean_sim, std_sim


# Apply to all 817 questions
results = []
for question, group in raw.groupby('Question'):
    indices = group.index.tolist()
    if len(indices) < 2:
        continue
    mean_sim, std_sim = compute_inconsistency(indices, embeddings)
    results.append({
        'Question':           question,
        'LSA_Sim_Mean':       mean_sim,
        'LSA_Sim_Std':        std_sim,
        'LSA_Inconsistency':  1 - mean_sim,
    })

lsa_df = pd.DataFrame(results)
print(f"Computed for {len(lsa_df)} questions")
print()
print(lsa_df[['LSA_Sim_Mean', 'LSA_Inconsistency']].describe().round(3))

## 5. Quick Sanity Check: Most vs Least Consistent Questions

Before doing any formal analysis, let's look at actual examples to make sure the scores make intuitive sense.

In [ ]:
# Most consistent (LLM gives very similar answers at all temperatures)
print("=" * 65)
print("TOP 5 MOST CONSISTENT QUESTIONS (low inconsistency)")
print("=" * 65)
top_consistent = lsa_df.nsmallest(5, 'LSA_Inconsistency')
for _, row in top_consistent.iterrows():
    print(f"\nQ: {row['Question']}")
    print(f"   LSA inconsistency: {row['LSA_Inconsistency']:.4f}")
    # Show the 5 answers
    answers = raw[raw['Question'] == row['Question']].sort_values('Temperature')['Answer'].tolist()
    for i, ans in enumerate(answers[:2]):  # show first 2 for brevity
        print(f"   Temp {[0.1,0.4,0.7,1.0,1.3][i]}: {ans[:80]}...")

print()
print("=" * 65)
print("TOP 5 MOST INCONSISTENT QUESTIONS (high inconsistency)")
print("=" * 65)
top_inconsistent = lsa_df.nlargest(5, 'LSA_Inconsistency')
for _, row in top_inconsistent.iterrows():
    print(f"\nQ: {row['Question']}")
    print(f"   LSA inconsistency: {row['LSA_Inconsistency']:.4f}")
    answers = raw[raw['Question'] == row['Question']].sort_values('Temperature')['Answer'].tolist()
    for i, ans in enumerate(answers[:2]):
        print(f"   Temp {[0.1,0.4,0.7,1.0,1.3][i]}: {ans[:80]}...")

## 6. Merge with Categories and Compare Signals

In [ ]:
# Merge LSA results with preprocessed data (which has categories + TF-IDF baseline)
df = processed.merge(lsa_df, on='Question', how='inner')
print(f"Final dataset: {len(df)} questions")
print(f"Columns: {df.columns.tolist()}")

# Correlation between the two signals
corr = df['Inconsistency_Score'].corr(df['LSA_Inconsistency'])
print(f"\nCorrelation between TF-IDF and LSA signals: r = {corr:.3f}")
print("(r=0.7 means they agree directionally but aren't redundant — good)")

## 7. Validation: Do Risk Tiers Rank Correctly?

This is the core validation. If our signals are capturing hallucination risk, then:

**High risk categories** (Conspiracies, Paranormal, Fiction) → should score **higher** inconsistency  
**Lower risk categories** (Science, Health) → should score **lower** inconsistency

In [ ]:
tier_order = ['High', 'Medium', 'Lower']
tier_df = df[df['Risk_Tier'].isin(tier_order)].copy()

tier_stats = (
    tier_df.groupby('Risk_Tier')[['Inconsistency_Score', 'LSA_Inconsistency']]
    .agg(['mean', 'median', 'std'])
    .loc[tier_order]
)
print("Inconsistency scores by risk tier:")
print(tier_stats.round(4).to_string())
print()
print("✓ Both signals rank: High > Medium > Lower")
print("✓ LSA separates the tiers more cleanly than TF-IDF")

In [ ]:
# Category-level breakdown
cat_stats = (
    tier_df.groupby(['Category', 'Risk_Tier'])[['Inconsistency_Score', 'LSA_Inconsistency']]
    .mean()
    .reset_index()
    .sort_values('LSA_Inconsistency', ascending=False)
)
print("Categories ranked by LSA inconsistency (highest = most hallucination-prone):")
print(cat_stats[['Category', 'Risk_Tier', 'LSA_Inconsistency']].to_string(index=False))

## 8. Visualization

In [ ]:
tier_colors = {'High': '#E8956D', 'Medium': '#7AAEC8', 'Lower': '#8DB89A'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('FACTS: Consistency Signals by Hallucination Risk Tier', 
             fontsize=13, fontweight='bold', y=1.02)

# --- Plot 1: TF-IDF baseline by tier ---
data_tfidf = [tier_df[tier_df['Risk_Tier']==t]['Inconsistency_Score'].dropna() for t in tier_order]
bp1 = axes[0].boxplot(data_tfidf, tick_labels=[f'{t}\nRisk' for t in tier_order], patch_artist=True)
for patch, t in zip(bp1['boxes'], tier_order):
    patch.set_facecolor(tier_colors[t]); patch.set_alpha(0.75)
axes[0].set_title('Signal 1: TF-IDF Baseline', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Inconsistency Score (1 - similarity)', fontsize=10)

# --- Plot 2: LSA embeddings by tier ---
data_lsa = [tier_df[tier_df['Risk_Tier']==t]['LSA_Inconsistency'].dropna() for t in tier_order]
bp2 = axes[1].boxplot(data_lsa, tick_labels=[f'{t}\nRisk' for t in tier_order], patch_artist=True)
for patch, t in zip(bp2['boxes'], tier_order):
    patch.set_facecolor(tier_colors[t]); patch.set_alpha(0.75)
axes[1].set_title('Signal 2: LSA Semantic Embeddings', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Inconsistency Score (1 - similarity)', fontsize=10)

# --- Plot 3: Scatter — TF-IDF vs LSA ---
for tier in tier_order:
    sub = tier_df[tier_df['Risk_Tier'] == tier]
    axes[2].scatter(sub['Inconsistency_Score'], sub['LSA_Inconsistency'],
                    color=tier_colors[tier], alpha=0.45, s=18, label=f'{tier} Risk')
axes[2].set_xlabel('TF-IDF Inconsistency', fontsize=10)
axes[2].set_ylabel('LSA Inconsistency', fontsize=10)
axes[2].set_title(f'Signal Correlation  r = {corr:.2f}', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig4_embedding_results.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Category-level bar chart (LSA signal)
cat_plot = cat_stats.sort_values('LSA_Inconsistency')

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    cat_plot['Category'],
    cat_plot['LSA_Inconsistency'],
    color=[tier_colors[t] for t in cat_plot['Risk_Tier']],
    edgecolor='white', linewidth=0.5
)
ax.set_xlabel('Mean LSA Inconsistency Score', fontsize=11)
ax.set_title('Hallucination Risk by Category (LSA Signal)', fontsize=13, fontweight='bold')

legend_elements = [
    mpatches.Patch(facecolor=tier_colors[t], label=f'{t} Risk') 
    for t in ['High', 'Medium', 'Lower']
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('fig5_category_lsa.png', dpi=130, bbox_inches='tight')
plt.show()

## 9. Interesting Finding: Misquotations

Misquotations score very **low** inconsistency despite being a "High risk" category. This is worth examining — it means the model confidently gives the same wrong answer at all temperatures. This is a different failure mode: **consistent hallucination**, not inconsistent hallucination.

In [ ]:
misquotes = df[df['Category'] == 'Misquotations'].sort_values('LSA_Inconsistency')
print("Misquotation questions — all score LOW inconsistency:")
print(misquotes[['Question', 'LSA_Inconsistency', 'Inconsistency_Score']].to_string(index=False))
print()
print("Why? The model has memorized a wrong attribution and repeats it confidently.")
print("This shows our signal catches UNCERTAIN hallucination, not CONFIDENT hallucination.")
print("The labeled holdout (professor's suggestion) will help capture both failure modes.")

## 10. Save Final Dataset

In [ ]:
# Combined dataset with both signals
df.to_csv('truthfulqa_signals.csv', index=False)

print("Saved: truthfulqa_signals.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns:")
for col in df.columns:
    print(f"  {col}")

## Summary

| Step | Result |
|------|--------|
| LSA embeddings (100 dims) | 25.6% variance explained |
| Signal correlation (TF-IDF vs LSA) | r = 0.71 |
| Risk tier ranking (both signals) | High > Medium > Lower ✓ |
| Interesting finding | Misquotations = consistent but wrong |

### Next steps
1. Replace LSA with a transformer sentence encoder (`all-MiniLM-L6-v2`) for stronger embeddings
2. Add NLI pairwise contradiction scoring (Signal 3)
3. Build labeled holdout set (~50–100 questions) for proper evaluation
4. Combine signals and report precision/recall/AUC against labels